# Μῆτις (Metis) — From-Scratch Language Model

<img src="https://img.shields.io/badge/Python-3.10+-blue.svg">
<img src="https://img.shields.io/badge/PyTorch-2.0+-ee4c2c.svg">

**Train a real decoder-only transformer (RMSNorm + RoPE + SwiGLU + GQA) on your own data — on a single GPU or even just a CPU.**

New in v2.5: **Grouped Query Attention (GQA)** for faster inference (same approach as LLaMA 2/3), multi-file dataset loading, and a rich **Siraj-ud-Daulah** knowledge dataset.

This notebook runs end-to-end: install → download/create data → train → chat.
Click **Runtime → Run all** (or press `Ctrl+F9`) to run everything.

## 1. Install Dependencies

In [ ]:
!pip install -q torch tqdm numpy matplotlib 2>&1 | tail -5

## 2. Import the Metis Package

The `metis/` directory should be in the working directory when this notebook is run. \
If you cloned the repo, you're all set — the package is importable directly.

In [ ]:
import sys, os
# Ensure we can find the metis package (adjust path for Colab if needed)
sys.path.insert(0, os.path.abspath('.'))

from metis import (
    MetisLM, ModelConfig, CharTokenizer,   # core
    PRESETS, setup_logging,                # config
    generate_text, load_model_and_tokenizer,  # generation
    train,                                  # training
)
print('Metis imported successfully ✓')
print(f"GQA available: use n_kv_heads < n_heads for grouped query attention")

## 3. Prepare the Dataset

### Option A: Tiny Shakespeare (character-level, ~1.1 MB)
A classic LM benchmark.

In [ ]:
import urllib.request

os.makedirs('data', exist_ok=True)
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'data/input.txt')
size = os.path.getsize('data/input.txt')
print(f'Tiny Shakespeare downloaded ({size:,} bytes)')

### Option B: Conversational Dataset (for a chat-style model)
Generates a simple "User: Hello / Metis: Hi" style dataset. Useful if you want the model to mimic a dialog agent quicky.

In [ ]:
conversations = [
    "User: Hello\nMetis: Hello! How can I help you today?\n",
    "User: Hi\nMetis: Hey! What's up?\n",
    "User: Who are you?\nMetis: I am Metis, a tiny AI assistant.\n",
    "User: What is your name?\nMetis: My name is Metis.\n",
    "User: Good morning\nMetis: Good morning! How are you?\n",
    "User: How are you?\nMetis: I'm doing great, thanks for asking!\n",
    "User: What do you do?\nMetis: I chat with you and answer questions.\n",
    "User: Tell me a joke\nMetis: I'm still learning how to be funny!\n",
    "User: Thanks\nMetis: You're welcome!\n",
    "User: Bye\nMetis: Goodbye! Have a great day.\n",
    "User: Who made you?\nMetis: I was created by you!\n",
    "User: Are you an AI?\nMetis: Yes, I am a tiny language model named Metis.\n",
    "User: Howdy\nMetis: Howdy! What's on your mind?\n",
]

with open('data/conversations.txt', 'w', encoding='utf-8') as f:
    for _ in range(500):
        f.writelines(conversations)

size = os.path.getsize('data/conversations.txt')
print(f'Conversational dataset created ({size:,} bytes)')

In [ ]:
---
## 4. Train the Model

Use the built-in `metis train` command or call `train()` directly from Python.

**Model presets:**

| Preset  | d_model | Heads | KV Heads | Layers | Seq Len | ~Params | VRAM   |
|---------|---------|-------|----------|--------|---------|---------|--------|
| `tiny`  | 128     | 4     | 2 (GQA)  | 4      | 256     | ~0.8M   | <1 GB  |
| `small` | 256     | 4     | 4 (MHA)  | 4      | 256     | ~4M     | ~1 GB  |
| `medium`| 384     | 6     | 6 (MHA)  | 6      | 512     | ~15M    | ~3 GB  |
| `large` | 512     | 8     | 8 (MHA)  | 8      | 512     | ~35M    | ~6 GB  |

**GQA (Grouped Query Attention):** Pass `--n-kv-heads N` to use fewer KV heads than query heads (e.g. `--n-kv-heads 2` with 4 query heads). This matches the approach used in LLaMA 2/3 for efficient inference. The KV-cache shrinks proportionally, and the `_repeat_kv` operation expands them at runtime.

**Multi-file datasets:** Pass a directory path instead of a file — all `.txt` files will be concatenated:
```bash
metis train --dataset data/ --preset tiny --n-kv-heads 2
```

# Build the config from a preset, with optional overrides.
# Use --n-kv-heads to enable Grouped Query Attention (GQA).
# GQA stores fewer key/value heads = smaller KV-cache, faster inference.

config = ModelConfig.from_preset(
    "tiny",                # fastest; try "small" or "medium" for better quality
    dataset_path="data/siraj_all.txt",  # ← change to your dataset
    max_iters=2000,         # increase for better convergence (5000+ recommended)
    log_level="INFO",
    n_kv_heads=2,           # GQA: 2 KV heads (vs 4 query heads). 0 = standard MHA.
)
print(config.summary())
print(f"GQA: {config.n_heads} query heads × {config.n_kv_heads} KV heads ({config.n_groups} groups)")

---
## 4. Train the Model

Use the built-in `metis train` command or call `train()` directly from Python.

**Model presets:**

| Preset  | d_model | Heads | Layers | Seq Len | ~Params | VRAM   |
|---------|---------|-------|--------|---------|---------|--------|
| `tiny`  | 128     | 4     | 4      | 256     | ~1M     | <1 GB  |
| `small` | 256     | 4     | 4      | 256     | ~4M     | ~1 GB  |
| `medium`| 384     | 6     | 6      | 512     | ~15M    | ~3 GB  |
| `large` | 512     | 8     | 8      | 512     | ~35M    | ~6 GB  |

In [ ]:
# Build the config from a preset, with optional overrides.
# Change the name or tweak the values below.
config = ModelConfig.from_preset(
    "tiny",                # fastest; try "small" or "medium" for better quality
    dataset_path="data/input.txt",
    max_iters=2000,         # increase for better convergence (5000+ recommended)
    log_level="INFO",
)
print(config.summary())

In [ ]:
# Train! This prints progress with a live progress bar and periodic validation.
train(config, resume=False)

---
## 5. Generate Text

Load the trained model and generate text in three ways.

In [ ]:
# Load the best model from checkpoints
model, tokenizer, gen_config = load_model_and_tokenizer('checkpoints')

### Single Prompt

In [ ]:
prompt = "O Romeo, Romeo! wherefore art thou Romeo?"
print(f"Prompt: {prompt}")
print("─" * 50)
output = generate_text(
    model, tokenizer, prompt,
    max_new_tokens=300,
    temperature=0.8, top_k=40, top_p=0.9,
    device=gen_config.device,
)
print(output[len(prompt):])

### Interactive Chat (in this notebook)

Run the cell below and type your messages. Commands:
- `/quit` or `/exit` — end
- `/clear` — reset history
- `/temp 0.5` — adjust temperature
- `Ctrl+C` — interrupt in Colab

In [ ]:
from metis import chat as interactive_chat
interactive_chat(model, tokenizer, gen_config, stream=True)

---
## 6. Checkpoint Status

List saved model artifacts and see the training config.

In [ ]:
import os
ckpt_dir = 'checkpoints'
print(f"Checkpoint directory: {os.path.abspath(ckpt_dir)}\n")
for name in sorted(os.listdir(ckpt_dir)):
    path = os.path.join(ckpt_dir, name)
    size = os.path.getsize(path)
    size_str = f"{size/1e6:.1f} MB" if size >= 1e6 else f"{size:,} bytes"
    print(f"  {name:<30} {size_str}")

---

*Built from scratch — RMSNorm, RoPE, SwiGLU, KV-cache, gradient checkpointing, no shortcuts.*

**Μῆτις** — *wisdom through craft*